# 00. Dataset preparation - GSE69914

### Libraries

In [2]:
!pip -q install GEOparse polars pyarrow


In [3]:
import GEOparse
import pandas as pd
import polars as pl
import os, re, csv
import pyarrow as pa
import pyarrow.csv as pacsv
import pyarrow.parquet as pq
import numpy as np


## 1. Fetch numeric labels from GEO (0..4)

In [ ]:
# === Cell 1: fetch numeric labels (0..4) from GEO ===
LABELS_CSV = "/kaggle/working/GSE69914_labels_numeric.csv"

# Download + parse GEO entry (lightweight vs matrix)
gse = GEOparse.get_GEO("GSE69914", destdir=".")

rows = []
for gsm_name, gsm in gse.gsms.items():
    txt = " | ".join([f"{k}: {v}" for k, v in gsm.metadata.items()]).lower()
    m = re.search(r"status\s*\([^)]*\)\s*:\s*([0-4])", txt)  # status(...): X
    code = int(m.group(1)) if m else None
    rows.append({"gsm": gsm_name, "status_code": code})

labels_df = pd.DataFrame(rows, columns=["gsm", "status_code"])
labels_df.to_csv(LABELS_CSV, index=False)

print(f"✅ Labels saved: {LABELS_CSV}")
print(labels_df["status_code"].value_counts(dropna=False).sort_index())


> ```
> ✅ Labels saved: /kaggle/working/GSE69914_labels_numeric.csv
> status_code
> 0      50
> 1      42
> 2     305
> 3       7
> 4       3
> Name: count, dtype: int64
> ```

## 2. Read TXT → transpose to Sample×CpG → attach labels → write Parquet (LZ4)

In [ ]:
# === Cell 2: TXT -> (transpose) -> attach labels -> Parquet (LZ4) ===
INPUT_TXT  = "/kaggle/input/gse69914-series-matrix-txt/GSE69914_series_matrix.txt"  # change if needed
LABELS_CSV = "/kaggle/working/GSE69914_labels_numeric.csv"
OUT_PAR    = "/kaggle/working/GSE69914_beta_with_labels_numeric_lz4.parquet"

assert os.path.exists(INPUT_TXT),  f"Missing: {INPUT_TXT}"
assert os.path.exists(LABELS_CSV), f"Missing: {LABELS_CSV} (run Cell 1 first)"

# 1) Load labels into a pandas Series for fast mapping
lab = pd.read_csv(LABELS_CSV)
lab = lab.dropna(subset=["gsm"]).copy()
lab["gsm"] = lab["gsm"].astype(str)
lab_series = pd.Series(lab["status_code"].astype("Int8").values, index=lab["gsm"].values)

# 2) Read the GEO Series Matrix (tab-delimited), skip header metadata
#    We load as float then downcast to float32 to save RAM.
df = pd.read_csv(
    INPUT_TXT,
    sep="\t",
    skiprows=73,      # skip the 73-line GEO header
    comment="!",      # ignore trailing metadata lines
    engine="c",
    dtype=None,       # let pandas infer, we'll cast after
    low_memory=False,
)

# Ensure first column is ID_REF (CpG / probe ID)
if df.columns[0] != "ID_REF":
    df.rename(columns={df.columns[0]: "ID_REF"}, inplace=True)

# 3) Cast all sample columns to float32 (keep ID_REF as string)
sample_cols = df.columns.tolist()[1:]
for c in sample_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce").astype(np.float32)

# 4) Transpose to Sample x CpG (rows=samples, cols=probes)
df_T = df.set_index("ID_REF").T
df_T.index.name = "id_tissue"   # index are GSM sample IDs

# 5) Bring index to a column, attach numeric label, enforce compact dtypes
df_T = df_T.reset_index()                 # id_tissue becomes a column
df_T.insert(1, "label", lab_series.reindex(df_T["id_tissue"]).values)  # insert as 2nd column
df_T["label"] = df_T["label"].astype("Int8")

# 6) Convert to Arrow Table with explicit schema and write Parquet (LZ4)
#    Build fields: id_tissue=utf8, label=int8, all probe columns=float32
fields = [pa.field("id_tissue", pa.string()),
          pa.field("label", pa.int8())]
for col in df_T.columns[2:]:
    fields.append(pa.field(col, pa.float32()))
schema = pa.schema(fields)

# Create Arrow Table without copying data when possible
table = pa.Table.from_pandas(df_T, preserve_index=False, schema=schema)

# Write Parquet with LZ4 (fast & lossless). Dictionary encoding helps metadata.
pq.write_table(
    table,
    OUT_PAR,
    compression="lz4",
    use_dictionary=True
)

print(f"✅ Parquet saved: {OUT_PAR}")

# 7) Quick sanity checks: shape and a tiny peek (fast)
par = pl.read_parquet(OUT_PAR)          # full read (ensure it loads)
print("Parquet shape:", par.shape)      # expected ~ (407, ~485k + 2)
print(par.select(["id_tissue","label"]).head())


> ```
> ✅ Parquet saved: /kaggle/working/GSE69914_beta_with_labels_numeric_lz4.parquet
> Parquet shape: (407, 485514)
> shape: (5, 2)
> ┌────────────┬───────┐
> │ id_tissue  ┆ label │
> │ ---        ┆ ---   │
> │ str        ┆ i8    │
> ╞════════════╪═══════╡
> │ GSM1712367 ┆ 2     │
> │ GSM1712368 ┆ 1     │
> │ GSM1712369 ┆ 0     │
> │ GSM1712370 ┆ 2     │
> │ GSM1712371 ┆ 2     │
> └────────────┴───────┘
> ```

## 3. Ultra-Fast Future Reading (Polars)

In [ ]:
PAR = "/kaggle/input/gse69914-parquet/GSE69914.parquet"

# Optional: keep console output readable (does NOT limit what you load)
pl.Config.set_tbl_rows(10)
pl.Config.set_tbl_cols(30)

# Load the full dataset (all rows, all columns)
try:
    df = pl.read_parquet(PAR)   # fast full load
except Exception as e:
    # Fallback if schema metadata is huge
    import pyarrow as pa
    table = pq.read_table(
        PAR,
        thrift_string_size_limit = 1 << 31,
        thrift_container_size_limit = 1 << 31
    )
    df = pl.from_arrow(table)

# 1) Rows/columns count
print("Shape (rows, cols):", df.shape)

# 2) Show first 5 rows of the first 5 columns
first5_cols = df.columns[:5]
print("Preview of first 5 columns:", first5_cols)
print(df.select(first5_cols).head(5))

# Example: row-wise mean across all probes (excluding id_tissue/label), but only display first 5 rows
row_mean_first5 = (
    df
    .with_columns(
        row_mean = pl.mean_horizontal(pl.exclude(["id_tissue", "label"]))
    )
    .select(["id_tissue", "label", "row_mean"])
    .head(5)
)
print(row_mean_first5)

print("✅ All columns are present and readable; full-dataset operations are supported.")


> ```
> Shape (rows, cols): (407, 485514)
> Preview of first 5 columns: ['id_tissue', 'label', 'cg00000029', 'cg00000108', 'cg00000109']
> shape: (5, 5)
> ┌────────────┬───────┬────────────┬────────────┬────────────┐
> │ id_tissue  ┆ label ┆ cg00000029 ┆ cg00000108 ┆ cg00000109 │
> │ ---        ┆ ---   ┆ ---        ┆ ---        ┆ ---        │
> │ str        ┆ i8    ┆ f32        ┆ f32        ┆ f32        │
> ╞════════════╪═══════╪════════════╪════════════╪════════════╡
> │ GSM1712367 ┆ 2     ┆ 0.258254   ┆ 0.986116   ┆ 0.889916   │
> │ GSM1712368 ┆ 1     ┆ 0.197553   ┆ 0.981426   ┆ 0.82683    │
> │ GSM1712369 ┆ 0     ┆ 0.275187   ┆ 0.972137   ┆ 0.839431   │
> │ GSM1712370 ┆ 2     ┆ 0.150849   ┆ 0.984434   ┆ 0.950852   │
> │ GSM1712371 ┆ 2     ┆ 0.240538   ┆ 0.987393   ┆ 0.897285   │
> └────────────┴───────┴────────────┴────────────┴────────────┘
> shape: (5, 3)
> ┌────────────┬───────┬──────────┐
> │ id_tissue  ┆ label ┆ row_mean │
> │ ---        ┆ ---   ┆ ---      │
> │ str        ┆ i8    ┆ f32      │
> ╞════════════╪═══════╪══════════╡
> │ GSM1712367 ┆ 2     ┆ 0.530135 │
> │ GSM1712368 ┆ 1     ┆ 0.509796 │
> │ GSM1712369 ┆ 0     ┆ 0.515661 │
> │ GSM1712370 ┆ 2     ┆ 0.500214 │
> │ GSM1712371 ┆ 2     ┆ 0.542685 │
> └────────────┴───────┴──────────┘
> ✅ All columns are present and readable; full-dataset operations are supported.
> ```